## HyperParameter Optimzation for Keras with Scikit-Learn

In [19]:
from tqdm import tqdm
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
import keras
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense, Dropout, Input
from keras.datasets import reuters
from scikeras.wrappers import KerasClassifier

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

### Defining Helper Functions

In [7]:
# vectorizing integer sequence
def vectorize_sequence(sequence, dims):
    results = np.zeros((len(sequence), dims))

    for idx, value in enumerate(sequence):
        if max(value) < dims:
            results[idx, value] = 1

    return results

# converts label in one-hot format
def one_hot_label(labels, dims):
    results = np.zeros((len(labels), dims))

    for idx, value in enumerate(labels):
        if value < dims:
            results[idx, value] = 1

    return results

In [12]:
# load the dataset
df = reuters.load_data(
    path="reuters.npz",
    num_words=None,
    skip_top=0,
    maxlen=None,
    test_split=0.2,
    seed=113,
    start_char=1,
    oov_char=2,
    index_from=3,
)

(X_train, y_train), (X_test, y_test) = reuters.load_data(
    path="reuters.npz",
    num_words=None,
    skip_top=0,
    maxlen=None,
    test_split=0.2,
    seed=113,
    start_char=1,
    oov_char=2,
    index_from=3,
)

word_to_ind = reuters.get_word_index(path="reuters_word_index.json")

dim_x = max([max(sequence) for sequence in X_train]) + 1
dim_y = max(y_train) + 1

X_train_vec = vectorize_sequence(X_train, dim_x)
X_test_vec = vectorize_sequence(X_test, dim_x)
Y_train_hot = one_hot_label(y_train, dim_y)
y_test_hot = one_hot_label(y_test, dim_y)

In [22]:
def create_model(neurons = 10):
    model = Sequential()
    model.add(Input(shape=(X_train_vec.shape[1],)))
    model.add(Dense(neurons, activation='linear'))
    model.add(Dense(64, activation='relu'))
    model.add(Dense(46, activation='softmax'))
    model.compile(optimizer='RMSprop', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

In [24]:
np.random.seed(0)

base_model = KerasClassifier(model=create_model, verbose=0, batch_size=10, epochs=1)
base_model.fit(X_train_vec, Y_train_hot)
base_score = base_model.score(X_test_vec, y_test_hot)
print(f"Pre-tuned Baseline Accuracy: {base_score:.3f}")

Pre-tuned Baseline Accuracy: 0.716


## Randomized Search

In [31]:
# parameters:
batch_sizes = [10, 20, 60, 80]
epochs = [1, 3, 5]
neurons = [1, 10, 20, 30]

params = dict(batch_size=batch_sizes, epochs=epochs, model__neurons=neurons)
print("Parameter Search Space:")
params

Parameter Search Space:


{'batch_size': [10, 20, 60, 80],
 'epochs': [1, 3, 5],
 'model__neurons': [1, 10, 20, 30]}

In [32]:
search = RandomizedSearchCV(estimator=base_model, param_distributions=params,
                            cv=3)
search_result = search.fit(X_train_vec, Y_train_hot)
best_params = search.best_params_

means = search_result.cv_results_['mean_test_score']
std = search_result.cv_results_['std_test_score']

print(f"Best Mean Cross-validated Score: {search_result.best_score_:.3f} using: {best_params}")

Best Mean Cross-validated Score: 0.799 using: {'model__neurons': 30, 'epochs': 5, 'batch_size': 80}


In [33]:
for mean, stds, param in zip(means, std, best_params):
    print("Mean cross-validated score: {} ({}) using: {}".format(round(mean,3), round(stds,3), param))

Mean cross-validated score: 0.381 (0.009) using: model__neurons
Mean cross-validated score: 0.799 (0.004) using: epochs
Mean cross-validated score: 0.723 (0.005) using: batch_size


In [34]:
print("Best test score: %.3f" % search_result.best_estimator_.score(X_test_vec, y_test_hot))

Best test score: 0.788
